# Сводка: дефицит потока или ошибка отклика

**Начните отсюда.** Этот ноутбук — короткий обзор; подробности каждого теста в
отдельных ноутбуках, ссылки в конце каждого раздела.

## Задача

Ню-классификатор — нейросеть, решающая по набору сработавших оптических модулей,
похоже ли событие на нейтрино. На экспериментальных данных она принимает
примерно **втрое больше** событий, чем предсказывает симуляция атмосферных
мюонов — единственного фона, который мы умеем считать. Это расхождение здесь
называется избытком.

Прежняя работа (`../excess_mechanism`) установила три вещи. Принятые
экспериментальные события — это катастрофически плохо реконструированные мюоны
того же рода, что симуляция производит. Ни одно отдельное доменное свойство —
заряды, шум, наклон струн, смещения времён — избытка не объясняет. И даже полный
многомерный доменный сдвиг не объясняет: приведение симуляции к признаковому
распределению эксперимента сдвигает верхнюю полосу лишь с 3.11 до 2.62.

## Что осталось, и почему это важно для статьи

**(a) дефицит потока.** Генератор недопроизводит какие-то конфигурации мюонов.
Отклик детектора при этом верен, симуляция нейтрино не затронута, и измеренная
на ней эффективность **переносится** на данные. Сломана только оценка фона.

**(b) ошибка отклика.** Реальные события труднее реконструируются, чем
симулированные. Тогда тот же механизм, что разворачивает нисходящие мюоны вверх,
разворачивал бы настоящие восходящие нейтрино вниз, и эффективность **не
переносится** — то есть под сомнением оказывается главный результат работы.

## Почему нужны специальные тесты

Всё, что мы видим, раскладывается так:

```
P(хиты)  =  P(хиты | трек)  ×  P(трек)
```

Гипотеза (a) портит второй множитель, (b) — первый. Любое сравнение обычных
распределений видит произведение и различить их не может. Каждый тест ниже либо
фиксирует трек, либо фиксирует поток.

Гипотезы и **опровергающие наблюдения записаны в `PROTOCOL.md` до запусков** и не
правились под результат.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(HERE / "src"))
import present

DATA = HERE / "data"
bound = pd.read_parquet(DATA / "20_achievable_bound.parquet")
cells = pd.read_parquet(DATA / "31_matched_cells.parquet")
bands = pd.read_parquet(DATA / "31_by_band.parquet")
scan = pd.read_parquet(DATA / "40_jitter_summary.parquet")
calib = pd.read_parquet(DATA / "41_matched_calibration.parquet")
partial = pd.read_parquet(DATA / "10_run_noise_partial.parquet")
observed = float(bands[bands.band_lo == 0.8].excess_in_band.iloc[0])
present.glossary(["скор ξ", "избыток", "полоса", "отбор качества"])
print(f"\nнаблюдаемый избыток при скоре выше 0.8: {observed:.2f}")

        скор ξ  —  выход классификатора от 0 до 1; чем выше, тем больше событие похоже на
                   нейтрино по мнению сети
       избыток  —  во сколько раз доля экспериментальных событий в данной области скора больше
                   такой же доли у симулированных мюонов. Единица означала бы согласие
        полоса  —  интервал скора, например 0.8–0.9
отбор качества  —  не меньше восьми сигнальных хитов не менее чем на трёх струнах (в коде это
                   условие называется h8s3, n_sn_hits ≥ 8 и n_sn_strings ≥ 3). Применяется к
                   обеим выборкам одинаково, иначе сравнивать было бы нечего

наблюдаемый избыток при скоре выше 0.8: 2.87


## Тест 4 — виноват ли шум воды

Биолюминесценция меняет скорость счёта между ранами впятеро. Эта вариация есть
только в данных, подделать корреляцию с ней нельзя. Если избыток гонит шум, в
шумных ранах он должен быть больше.

In [2]:
level = partial[partial.target == "level"].pivot(
    index="proxy", columns="control", values="rho")[
    ["none", "cluster", "quality_frac", "both"]].reset_index()
level = present.rename_values(level, "proxy")
present.show(
    level,
    {"proxy": "чем меряем шум", "none": "сырая корреляция",
     "cluster": "с поправкой на кластер",
     "quality_frac": "с поправкой на знаменатель",
     "both": "с поправкой на оба"},
    rows="один способ померить шумовую нагрузку",
    source=present.artefact_note(DATA / "10_run_noise_partial.parquet"),
    note="Шумовая версия (b) предсказывает ПОЛОЖИТЕЛЬНУЮ корреляцию. Ни в одном "
         "столбце её нет — знак всюду отрицательный.",
    decimals=3)

Строка — один способ померить шумовую нагрузку.
Шумовая версия (b) предсказывает ПОЛОЖИТЕЛЬНУЮ корреляцию. Ни в одном столбце её нет — знак
всюду отрицательный.
Источник: стадия 10_run_noise, файл 10_run_noise_partial.parquet, 24 строки


чем меряем шум,сырая корреляция,с поправкой на кластер,с поправкой на знаменатель,с поправкой на оба
сырых хитов на событие,-0.542000,-0.446000,-0.454000,-0.179000
"доля хитов, отвергнутых фильтром",-0.473000,-0.393000,-0.356000,-0.056000
"скорость счёта, Гц",-0.360000,-0.436000,-0.143000,-0.037000


**Вывод:** предсказанного эффекта нет. Но заявить измеренный ноль нельзя —
контрольные переменные коллинеарны на −0.95, и при 29 ранах оси не разделяются.
Схема не обладает мощностью, а не «эффекта нет».

Подробнее: **`10_run_noise.ipynb`**

## Тест 2 — может ли ошибка потока это объяснить

Точная верхняя граница, а не результат подгонки: если вес зависит только от
истины, максимум достижимого усиления считается аналитически.

In [3]:
top = bound[(bound.score_lo >= 0.8) & (bound.weight_range.isin([2.0, 4.0, 10.0]))].copy()
top["band"] = top.apply(
    lambda r: f"{r.score_lo:.2f}–{min(r.score_hi, 1.0):.2f}", axis=1)
present.show(
    top.pivot(index="weight_range", columns="band",
              values="max_enhancement").reset_index(),
    {"weight_range": "допустимый размах веса R",
     **{b: f"полоса {b}" for b in sorted(top.band.unique())}},
    rows="одно ограничение на то, во сколько раз вес может меняться",
    source=present.artefact_note(DATA / "20_achievable_bound.parquet"),
    note="Сравнивать с наблюдаемым: 2.71 / 2.81 / 3.29 / 4.39 по этим полосам. "
         "При R = 2 недостижимо нигде, при R = 4 — в двух верхних полосах.",
    decimals=2)

Строка — одно ограничение на то, во сколько раз вес может меняться.
Сравнивать с наблюдаемым: 2.71 / 2.81 / 3.29 / 4.39 по этим полосам. При R = 2 недостижимо
нигде, при R = 4 — в двух верхних полосах.
Источник: стадия 20_flux_reweight, файл 20_achievable_bound.parquet, 24 строки


допустимый размах веса R,полоса 0.80–0.90,полоса 0.90–0.95,полоса 0.95–0.99,полоса 0.99–1.00
2.000000,1.760000,1.760000,1.760000,1.770000
4.000000,3.010000,3.050000,3.050000,3.140000
10.000000,5.990000,6.140000,6.090000,6.590000


**Вывод:** перевзвешивание потока правдоподобного размаха породить избыток не
может. Гладкая подгонка требует размаха ×131 и подавления хорошо измеренной
основной массы мюонов вдвое, и при этом ухудшает величины, в подгонке не
участвовавшие.

Оговорка: истинных осей было две, прицельного параметра у нас нет вовсе.

Подробнее: **`20_flux_reweight.ipynb`**

## Тест 1 — одинаков ли отклик детектора

Выбросить один хит, подогнать трек по остальным, предсказать выброшенный.
Сравнение при зафиксированном треке.

In [4]:
present.show(
    bands,
    {"band_lo": "полоса от", "iqr_exp": "ширина остатка exp, нс",
     "iqr_mc": "ширина остатка МК, нс", "iqr_ratio": "отношение ширин",
     "implied_extra_sigma_ns": "недостающий разброс МК, нс",
     "excess_in_band": "избыток в этой области"},
    rows="все события со скором от указанного значения и выше",
    source=present.artefact_note(DATA / "31_by_band.parquet"),
    note="Отклик различается во всех областях, включая нижнюю, где избыток равен "
         "единице. Разрыв ПЛОСКИЙ, пока избыток растёт втрое.",
    decimals=3)
print(f"при согласовании по геометрии exp шире МК в "
      f"{(cells.iqr_ratio > 1).sum()} ячейках из {len(cells)}")
print(f"недостающий разброс: медиана {cells.implied_extra_sigma_ns.median():.1f} нс "
      f"(приборный остаток по каналам, измеренный ранее: ~10 нс)")

Строка — все события со скором от указанного значения и выше.
Отклик различается во всех областях, включая нижнюю, где избыток равен единице. Разрыв ПЛОСКИЙ,
пока избыток растёт втрое.
Источник: стадия 31_response_compare, файл 31_by_band.parquet, 5 строк


полоса от,"ширина остатка exp, нс","ширина остатка МК, нс",отношение ширин,"недостающий разброс МК, нс",избыток в этой области
0.000000,62.143000,47.524000,1.308000,29.682000,1.000000
0.100000,57.931000,35.945000,1.612000,33.678000,1.591000
0.500000,44.650000,31.666000,1.410000,23.334000,2.190000
0.800000,45.173000,31.740000,1.423000,23.827000,2.870000
0.900000,45.670000,34.101000,1.339000,22.519000,3.097000


при согласовании по геометрии exp шире МК в 66 ячейках из 66
недостающий разброс: медиана 28.5 нс (приборный остаток по каналам, измеренный ранее: ~10 нс)


**Вывод:** отклик детектора смоделирован неверно — симуляции не хватает около
29 нс временного разброса. Но разрыв не растёт вместе с избытком, чего протокол
ожидал бы от (b).

Это (b) не снимает: доля принятых равна 3.4 × 10⁻⁴, и равномерное ухудшение,
сдвигающее доли процента событий, умножает хвост в разы, не трогая массу.

Подробнее: **`30_response.ipynb`**

## Тест 3 — воспроизводит ли измеренная ошибка избыток

Добавить в симуляцию ровно измеренные 29 нс и переиграть обе сети с сырых хитов.
Величина не подбиралась — она пришла из теста 1, по другой наблюдаемой.

In [5]:
present.show(
    scan,
    {"sigma_ns": "добавленный разброс σ, нс",
     "n_quality": "прошло отбор качества", "n_accepted": "принято",
     "acceptance": "доля принятых",
     "induced_excess": "во сколько раз выросла доля принятых"},
    rows="одна величина размытия",
    source=present.artefact_note(DATA / "40_jitter_summary.parquet"),
    decimals=6)
needed = calib.sigma_needed_ns
predicted = np.interp([needed.quantile(.25), needed.median(), needed.quantile(.75)],
                      scan.sigma_ns, scan.induced_excess)
print(f"размытие, нужное чтобы совпасть с данными (по ячейкам геометрии): "
      f"{needed.median():.1f} нс, квартили {needed.quantile(.25):.1f}"
      f"–{needed.quantile(.75):.1f}")
print(f"избыток, который оно создаёт: {predicted[1]:.2f} "
      f"(по квартилям {predicted[0]:.2f}–{predicted[2]:.2f})")
print(f"НАБЛЮДАЕМЫЙ избыток: {observed:.2f}")

Строка — одна величина размытия.
Источник: стадия 40_time_jitter, файл 40_jitter_summary.parquet, 5 строк


"добавленный разброс σ, нс",прошло отбор качества,принято,доля принятых,во сколько раз выросла доля принятых
0.000000,693009,239,0.000345,1.000000
15.000000,651787,337,0.000517,1.499219
25.000000,590731,478,0.000809,2.346276
30.000000,556375,538,0.000967,2.803856
40.000000,485336,665,0.001370,3.973014


размытие, нужное чтобы совпасть с данными (по ячейкам геометрии): 29.9 нс, квартили 18.3–37.9
избыток, который оно создаёт: 2.80 (по квартилям 1.78–3.73)
НАБЛЮДАЕМЫЙ избыток: 2.87


**Вывод:** измеренная независимо величина размытия воспроизводит наблюдаемый
избыток.

Подробнее: **`40_jitter.ipynb`**

---

# Итог

Три теста из четырёх сходятся на **(b) — ошибке отклика детектора**.

## Что показано

* Перевзвешивание потока правдоподобного размаха породить избыток **не способно**
  — это точная граница, а не результат оптимизации.
* Отклик детектора **действительно различается**, во всех сопоставленных ячейках
  геометрии; симуляции не хватает около 29 нс временного разброса.
* Размытие времён **измеренной** величины воспроизводит наблюдаемый избыток.

## Чего не показано

* Что размытие времён — **единственная** возможная причина. Тест 3 показывает
  достаточность, не единственность: гауссово размытие — это модель ошибки
  отклика, и другая модель похожего размера могла бы дать тот же результат.
  Каскадное объяснение ослаблено симметричностью, но не исключено, и
  одностороннее возмущение не проверялось.
* Совпадение не такое точное, как выглядит по центральным значениям: разброс
  калибровки предсказывает избыток от 1.8 до 3.7, и наблюдаемое лежит внутри.
  Это согласие, а не точное попадание.
* Прицельный параметр в тесте 2 недоступен; тест 4 не обладает мощностью.

## Что осталось необъяснённым

Размытие не выравнивает число хитов: у эксперимента медиана 10, у размытой
симуляции 12–13. Одним временным разбросом разница между выборками не
исчерпывается.

## Следствие для статьи

Пессимистическое, и обходить его формулировками нельзя. Если дело во временном
разрешении, реальные нейтрино размываются так же, и **измеренная на симуляции
эффективность отбора нейтрино не переносится на данные**. Читатель, который
сделает такой вывод из наблюдаемого избытка ложных срабатываний, будет прав.

Прямой способ закрыть вопрос — измерить эффективность **на данных**: взять из
`exp_reco` события, которые независимая стандартная реконструкция BARS помечает
как качественные восходящие треки, и сравнить долю принятых сетью с долей
принятых симулированных нейтрино сопоставимого качества.